# Classification from BrainVision through BIDS

This example generates eight tiny recordings, converts them to BIDS, constructs
labelled windows and feeds one fixed participant-held-out problem to established
classifiers. It downloads no data and uses no participant recordings.

Different source channel names and orders are aligned through an explicit mapping.
The mapping is known because the signals are generated here; removing prefixes
from an unknown recording would require acquisition evidence. Channels, rate and
window duration are choices for this example, not universal EEG requirements.

Install the development and notebook dependencies with
`pip install -e '.[binder,dev]'`. The development extra includes PyBV for generating
BrainVision fixtures. All generated files live in a temporary directory.

In [ ]:
from copy import deepcopy
from hashlib import sha256
from importlib.metadata import version
from pathlib import Path
from tempfile import TemporaryDirectory

import mne
import numpy as np
from aeon.classification.distance_based import KNeighborsTimeSeriesClassifier
from mne_bids import BIDSPath

from aeon_neuro.classification import CSPClassifier
from aeon_neuro.datasets import (
    build_classification_dataset,
    make_fixed_length_cases,
    validate_group_split,
)
from aeon_neuro.datasets.load_bids import load_bids_data
from aeon_neuro.utils.bids import convert_eeg_to_bids

workspace = TemporaryDirectory(prefix="aeon-neuro-classification-")
root = Path(workspace.name)
native_root = root / "native"
native_root.mkdir()
bids_root = root / "bids"

## Generate recordings with two known channel layouts

The four generated channels are in volts at 128 Hz. Each recording lasts eight
seconds. Alternating participants receive different synthetic spatial patterns.
The labels are an explicit table maintained separately from BIDS: a file name or
BIDS task entity does not determine a scientific prediction target.

One source layout uses ordinary names in a different order; the other uses
decorated names. Both measure the same generated signals. No interpolation,
re-referencing, filtering or resampling is needed for alignment.

In [ ]:
rng = np.random.default_rng(42)
channel_order = ["C3", "Cz", "C4", "Pz"]
sfreq = 128
time = np.arange(8 * sfreq) / sfreq
layouts = [
    [("C4", "C4"), ("C3", "C3"), ("Pz", "Pz"), ("Cz", "Cz")],
    [("Cz", "EEG Cz"), ("Pz", "EEG Pz"), ("C3", "EEG C3"), ("C4", "EEG C4")],
]
labels_by_subject = {}
source_names_by_subject = {}
encoded_inputs = {}

for person in range(8):
    subject = str(person + 1).zfill(2)
    label = person % 2
    layout = layouts[(person // 2) % 2]
    canonical = rng.normal(scale=5e-6, size=(4, len(time)))
    canonical[2 * label] += 8e-6 * np.sin(2 * np.pi * 10 * time)
    source_names = [name for _, name in layout]
    source_indices = [channel_order.index(site) for site, _ in layout]
    raw = mne.io.RawArray(
        canonical[source_indices],
        mne.create_info(source_names, sfreq, "eeg"),
        verbose=False,
    )
    native = native_root / f"person-{subject}.vhdr"
    raw.export(native, fmt="brainvision", verbose=False)
    # Compare later read-backs with the encoded source, since export quantises it.
    encoded_inputs[subject] = mne.io.read_raw_brainvision(
        native, verbose=False
    ).get_data()
    labels_by_subject[subject] = label
    source_names_by_subject[subject] = {site: name for site, name in layout}

source_hashes = {
    path: sha256(path.read_bytes()).hexdigest() for path in native_root.iterdir()
}

## Convert to BIDS and verify the source handoff

The public converter preserves available acquisition metadata. These generated
recordings have no electrode positions, acquisition-reference evidence or real
experimental events. They therefore demonstrate software integration rather than
complete acquisition metadata or independent structural BIDS validation.

In [ ]:
for subject in labels_by_subject:
    destination = BIDSPath(
        root=bids_root, subject=subject, task="synthetic", datatype="eeg"
    )
    convert_eeg_to_bids(native_root / f"person-{subject}.vhdr", destination)

recordings, metadata = load_bids_data(bids_root)
assert len(recordings) == len(metadata) == 8
for signal, info in zip(recordings, metadata):
    np.testing.assert_array_equal(signal, encoded_inputs[info["subject"]])
    assert info["sampling_frequency"] == sfreq
    # MNE reads these calibrated EEG amplitudes in volts.
    info["signal_unit"] = "V"

[(info["subject"], info["channel_names"]) for info in metadata[:4]]

## Construct cases using the explicit layout and target decisions

Choose one-second, nonoverlapping windows and retain every generated recording.
The existing case builder selects source names and gives the output a common
ordered identity. Original source names and sample intervals remain in each case.
Changing the classifier later will not change this case selection.

Real studies must justify the eligible interval, window length, target table and
mapping. A 19-channel profile can be supplied as another explicit selection when
all its sites are available; this example does not require it.

In [ ]:
labels = [labels_by_subject[info["subject"]] for info in metadata]
subjects = [info["subject"] for info in metadata]
cases = make_fixed_length_cases(
    recordings, metadata, labels=labels, groups=subjects, window_size=1
)
recording_info = {info["filepath"]: info for info in metadata}
for case in cases:
    subject = recording_info[case["recording_id"]]["subject"]
    case.update(
        participant_id=subject,
        channel_names=[
            source_names_by_subject[subject][site] for site in channel_order
        ],
        output_channel_names=channel_order,
    )

problem = build_classification_dataset(recordings, metadata, cases)
assert problem.X.shape == (64, 4, 128)
assert problem.metadata["channel_names"] == channel_order
for row, case in enumerate(problem.cases):
    info = recording_info[case["recording_id"]]
    picks = [info["channel_names"].index(name) for name in case["channel_names"]]
    expected = encoded_inputs[info["subject"]][
        picks, case["start_sample"] : case["stop_sample"]
    ]
    np.testing.assert_array_equal(problem.X[row], expected)

problem.X.shape, problem.metadata["channel_names"], problem.cases[0]

Missing sites and duplicate output identities fail rather than being silently
filled, dropped or merged. Automatic name cleanup would not establish whether a
channel is referential, bipolar or associated with a different participant.

In [ ]:
missing = deepcopy(cases)
missing[0]["channel_names"][0] = "unrecorded-site"
try:
    build_classification_dataset(recordings, metadata, missing)
except ValueError as error:
    assert "unknown source channels" in str(error)
else:
    raise AssertionError("A missing requested channel must fail.")

duplicate = deepcopy(cases)
duplicate[0]["output_channel_names"][1] = duplicate[0]["output_channel_names"][0]
try:
    build_classification_dataset(recordings, metadata, duplicate)
except ValueError as error:
    assert "unique strings" in str(error)
else:
    raise AssertionError("Duplicate output channel identities must fail.")

## Freeze a participant-held-out split

Participants 01–06 supply training cases; 07–08 supply test cases. This holds out
people, not random windows from people already seen during fitting. Both classes
are present in each partition. The conservative shared validator is appropriate
for this example; a within-person session experiment needs a different declared
policy. Actual case-ID assignments are retained alongside numeric row indices.

In [ ]:
train = np.flatnonzero(np.isin(problem.groups, ["01", "02", "03", "04", "05", "06"]))
test = np.flatnonzero(np.isin(problem.groups, ["07", "08"]))
validate_group_split(problem, train, test)
assert len(train) == 48 and len(test) == 16
assert set(problem.y[train]) == set(problem.y[test]) == {0, 1}
assignments = {
    name: [problem.cases[index]["case_id"] for index in indices]
    for name, indices in {"train": train, "test": test}.items()
}
before_fit = problem.X.copy()
{name: len(ids) for name, ids in assignments.items()}

## Feed the same problem to established classifiers

The ordinary aeon distance baseline and CSP+LDA both receive the same multichannel
arrays and split. CSP is supervised feature extraction inside its classifier, so
it learns only from training cases. Neither configuration performs internal
hyperparameter selection. If tuning is added, its inner folds must also respect
the intended training groups.

The outputs below check prediction alignment and probability columns. These toy
signals provide no evidence about real EEG accuracy or relative method quality.

In [ ]:
classifiers = {
    "euclidean_1nn": KNeighborsTimeSeriesClassifier(
        distance="euclidean", n_neighbors=1
    ),
    "csp_lda": CSPClassifier(
        n_components=2, reg="oas", cov_est="epoch", random_state=42
    ),
}
results = {}
for name, classifier in classifiers.items():
    classifier.fit(problem.X[train], problem.y[train])
    predicted = classifier.predict(problem.X[test])
    probabilities = classifier.predict_proba(problem.X[test])
    assert predicted.shape == (len(test),)
    assert probabilities.shape == (len(test), len(classifier.classes_))
    np.testing.assert_allclose(probabilities.sum(axis=1), 1)
    results[name] = {
        "case_ids": assignments["test"].copy(),
        "predictions": predicted.tolist(),
        "probability_classes": classifier.classes_.tolist(),
    }

np.testing.assert_array_equal(problem.X, before_fit)
assert all(
    sha256(path.read_bytes()).hexdigest() == digest
    for path, digest in source_hashes.items()
)
{
    name: {"cases": len(result["case_ids"]), "classes": result["probability_classes"]}
    for name, result in results.items()
}

Record package versions with the chosen construction, mapping and assignments
when keeping an experiment. Full reruns also need source identities and effective
pipeline settings. This notebook demonstrates the shared handoff and does not
define a stable experiment serialization format. Finally, remove generated files.

In [ ]:
versions = {
    package: version(package)
    for package in ("aeon-neuro", "aeon", "mne", "mne-bids", "scikit-learn")
}
workspace.cleanup()
versions